# Homework 2: Distributed LLM Training

**Model:** `EleutherAI/pythia-160m`  
**Dataset:** `wikitext-103-v1`  
**Metric:** Validation Perplexity (PPL)  
**Fixed:** `seed=42`, `seq_len=512`, `global_batch_size=131072` tokens/step, 1 epoch

---
## Инструкция по воспроизведению экспериментов

### Требования к окружению

```bash
# Python >= 3.10, PyTorch >= 2.4, CUDA >= 12.1
# Установка зависимостей через uv:
source .env
make vendor
```

Или вручную:
```bash
pip install torch transformers datasets accelerate
```

Модели и датасеты автоматически скачиваются с HuggingFace при первом запуске.  
Для кэширования на шаренное хранилище задаём переменные окружения:
```bash
export HF_HOME=/data/shared_ml/huggingface
export TRANSFORMERS_CACHE=/data/shared_ml/huggingface
```

---
### Task 1 — Single-GPU Baseline

Скрипт: `scripts/train_single.py`  
Каждую из трёх конфигураций запускать **отдельно**.

```bash
# fp32
CUDA_VISIBLE_DEVICES=0 uv run python scripts/train_single.py \
    --dtype fp32 --batch-size 8 \
    --experiment-name p1-fp32 --log-dir logs

# bf16
CUDA_VISIBLE_DEVICES=0 uv run python scripts/train_single.py \
    --dtype bf16 --batch-size 8 \
    --experiment-name p1-bf16 --log-dir logs

# bf16 + activation checkpointing
CUDA_VISIBLE_DEVICES=0 uv run python scripts/train_single.py \
    --dtype bf16 --activation-checkpointing --batch-size 8 \
    --experiment-name p1-bf16-ac --log-dir logs
```

Результаты сохраняются в `logs/p1-*/results.json` и `logs/p1-*/training_log.csv`.

---
### Task 2 — FSDP: стратегии шардирования (4 GPU)

Скрипт: `scripts/train_fsdp.py`  
Запускается через `torchrun`. Каждую стратегию запускать **отдельно**.

```bash
export OMP_NUM_THREADS=1
export TORCHELASTIC_ERROR_FILE=error.json

# NO_SHARD (реализован через DDP)
CUDA_VISIBLE_DEVICES=0,1,2,3 uv run torchrun \
    --nproc-per-node 4 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --strategy NO_SHARD --batch-size 4 \
    --experiment-name p2-no-shard --log-dir logs

# SHARD_GRAD_OP
CUDA_VISIBLE_DEVICES=0,1,2,3 uv run torchrun \
    --nproc-per-node 4 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --strategy SHARD_GRAD_OP --batch-size 4 \
    --experiment-name p2-shard-grad-op --log-dir logs

# FULL_SHARD
CUDA_VISIBLE_DEVICES=0,1,2,3 uv run torchrun \
    --nproc-per-node 4 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --strategy FULL_SHARD --batch-size 4 \
    --experiment-name p2-full-shard --log-dir logs
```

> **Важно:** `torchrun` при падении дочернего процесса иногда зависает, ожидая остальных workers.  
> Если скрипт завис после завершения обучения — это нормально: дождитесь таймаута NCCL (~2 мин)  
> или принудительно завершите через `Ctrl+C`. Результаты к тому моменту уже сохранены в `results.json`.

---
### Task 3 — CPU Offload (2 GPU)

```bash
export OMP_NUM_THREADS=1

# Baseline: FULL_SHARD без offload
CUDA_VISIBLE_DEVICES=0,1 uv run torchrun \
    --nproc-per-node 2 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --strategy FULL_SHARD --batch-size 4 \
    --experiment-name p3-full-shard --log-dir logs

# С CPUOffload
CUDA_VISIBLE_DEVICES=0,1 uv run torchrun \
    --nproc-per-node 2 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --strategy FULL_SHARD --cpu-offload --batch-size 4 \
    --experiment-name p3-cpu-offload --log-dir logs

# С CPUOffload + activation checkpointing
CUDA_VISIBLE_DEVICES=0,1 uv run torchrun \
    --nproc-per-node 2 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --strategy FULL_SHARD --cpu-offload \
    --activation-checkpointing --batch-size 4 \
    --experiment-name p3-cpu-offload-ac --log-dir logs
```

---
### Task 4 — Масштабирование по числу GPU

```bash
export OMP_NUM_THREADS=1

# 1 GPU — запускается напрямую через python (не torchrun)
CUDA_VISIBLE_DEVICES=0 uv run python scripts/train_fsdp.py \
    --strategy FULL_SHARD --batch-size 8 \
    --experiment-name p4-1gpu --log-dir logs

# 2 GPU
CUDA_VISIBLE_DEVICES=0,1 uv run torchrun \
    --nproc-per-node 2 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --strategy FULL_SHARD --batch-size 4 \
    --experiment-name p4-2gpu --log-dir logs

# 4 GPU
CUDA_VISIBLE_DEVICES=0,1,2,3 uv run torchrun \
    --nproc-per-node 4 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --strategy FULL_SHARD --batch-size 2 \
    --experiment-name p4-4gpu --log-dir logs
```

> **Примечание по `batch-size`:** при масштабировании `batch_size` уменьшается пропорционально числу GPU,
> чтобы сохранить фиксированный `global_batch_size = 131072` токенов/шаг:  
> `global_batch_size = seq_len × batch_size × n_gpus × grad_accum_steps = 512 × 8 × 1 = 512 × 4 × 2 = 512 × 2 × 4`

---
### Bonus A — pythia-410m, три стратегии (4 GPU)

```bash
export OMP_NUM_THREADS=1

# NO_SHARD
CUDA_VISIBLE_DEVICES=0,1,2,3 uv run torchrun \
    --nproc-per-node 4 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --model EleutherAI/pythia-410m \
    --strategy NO_SHARD --batch-size 2 \
    --experiment-name bonus-410m-no-shard --log-dir logs

# SHARD_GRAD_OP
CUDA_VISIBLE_DEVICES=0,1,2,3 uv run torchrun \
    --nproc-per-node 4 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --model EleutherAI/pythia-410m \
    --strategy SHARD_GRAD_OP --batch-size 2 \
    --experiment-name bonus-410m-shard-grad-op --log-dir logs

# FULL_SHARD
CUDA_VISIBLE_DEVICES=0,1,2,3 uv run torchrun \
    --nproc-per-node 4 --redirects 3 --log-dir logs \
    scripts/train_fsdp.py --model EleutherAI/pythia-410m \
    --strategy FULL_SHARD --batch-size 2 \
    --experiment-name bonus-410m-full-shard --log-dir logs
```

---
### Структура логов

После каждого запуска в `logs/<experiment-name>/` появляются:
- `results.json` — итоговые метрики (val PPL, peak memory, throughput)
- `training_log.csv` — пошаговые метрики (step, train_loss, tps, peak_mem_gb)
- `train.log` — текстовый лог процесса обучения

Notebook читает эти файлы из папки `logs/` автоматически.

---


In [ ]:
import json
import csv
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

os.makedirs('plots', exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
LOGS = Path('logs')

def load_results(exp_name: str) -> dict:
    """Load results.json for a given experiment."""
    p = LOGS / exp_name / 'results.json'
    if not p.exists():
        return {}
    with open(p) as f:
        return json.load(f)

def load_log(exp_name: str) -> list[dict]:
    """Load training_log.csv for a given experiment."""
    p = LOGS / exp_name / 'training_log.csv'
    if not p.exists():
        return []
    with open(p) as f:
        return list(csv.DictReader(f))

def fmt(r: dict, key: str, decimals: int = 2) -> str:
    v = r.get(key, None)
    if v is None:
        return '—'
    return f'{float(v):.{decimals}f}'

print('Utilities loaded.')

---
## 1. Single-GPU Baseline


In [ ]:
r_fp32  = load_results('p1-fp32')
r_bf16  = load_results('p1-bf16')
r_bf16ac = load_results('p1-bf16-ac')

print('Task 1 — Single-GPU Results')
print(f"{'Config':<35} {'Val PPL':>10} {'Peak Mem (GB)':>15} {'Throughput (tok/s)':>20}")
print('-' * 82)
for label, r in [('fp32', r_fp32), ('bf16', r_bf16), ('bf16 + activation checkpointing', r_bf16ac)]:
    print(f"{label:<35} {fmt(r,'val_ppl'):>10} {fmt(r,'peak_mem_gb'):>15} {fmt(r,'throughput_tps',0):>20}")

In [ ]:
# Plot training loss curves for Task 1
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, exp in [('fp32', 'p1-fp32'), ('bf16', 'p1-bf16'), ('bf16+AC', 'p1-bf16-ac')]:
    rows = load_log(exp)
    if rows:
        steps = [int(r['step']) for r in rows]
        losses = [float(r['train_loss']) for r in rows]
        tps_vals = [float(r['tps']) for r in rows]
        axes[0].plot(steps, losses, label=label)
        axes[1].plot(steps, tps_vals, label=label)

axes[0].set_title('Training Loss (Task 1)')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].set_title('Throughput tok/s (Task 1)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('tok/s')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/task1_curves.png', bbox_inches='tight')
plt.show()

### 1.A — Память: fp32 vs bf16 (2 балла)

**Теория.** Pythia-160m имеет ~160M параметров:
- **fp32**: 4 байта × 160M = **0.64 GB** только параметры  
- **bf16**: 2 байта × 160M = **0.32 GB** только параметры  
- Теоретическая разница: **0.32 GB**

**Реальная разница** (из экспериментов): **2.74 GB** (7.99 GB fp32 → 5.25 GB bf16)

**Почему реальная разница 2.74 GB >> 0.32 GB?**

При обучении в GPU хранятся не только параметры, но и:
1. **Градиенты** — в том же dtype, что и параметры (ещё ×1 от веса). Для fp32: +0.64 GB, для bf16: +0.32 GB → разница +0.32 GB.
2. **Optimizer states (Adam)** — в fp32 всегда (momentum + variance = ×2 × 4 байта). Для обеих точностей optimizer states ≈ одинаковы (~1.28 GB). Разницы нет.
3. **Активации** — при прямом проходе хранятся промежуточные тензоры (пропорциональны batch_size × seq_len × hidden_dim × n_layers). В fp32 они в **2× больше**, чем в bf16. Это главный источник разницы: ~2.1 GB экономии на активациях.

Суммарная реальная экономия:
$$\Delta_{\text{real}} = \underbrace{\Delta_{\text{params}}}_{0.32} + \underbrace{\Delta_{\text{grads}}}_{0.32} + \underbrace{\Delta_{\text{activations}}}_{\approx 2.1} \approx 2.74 \text{ GB}$$


In [ ]:
# 1.A — Compute and show memory breakdown
n_params = 160e6
fp32_params_gb = 4 * n_params / 1e9
bf16_params_gb = 2 * n_params / 1e9

mem_fp32 = r_fp32.get('peak_mem_gb', None)
mem_bf16 = r_bf16.get('peak_mem_gb', None)

print(f'Theoretical parameter memory fp32: {fp32_params_gb:.3f} GB')
print(f'Theoretical parameter memory bf16: {bf16_params_gb:.3f} GB')
print(f'Theoretical difference:            {fp32_params_gb - bf16_params_gb:.3f} GB')
if mem_fp32 and mem_bf16:
    real_diff = float(mem_fp32) - float(mem_bf16)
    print(f'\nMeasured peak mem fp32:            {float(mem_fp32):.3f} GB')
    print(f'Measured peak mem bf16:            {float(mem_bf16):.3f} GB')
    print(f'Real difference:                   {real_diff:.3f} GB')
    print(f'Overhead beyond params:            {real_diff - (fp32_params_gb - bf16_params_gb):.3f} GB')
    print('  → Explained by: gradients in matching dtype + activations in fp32 taking 2x more memory')

### 1.B — Throughput при activation checkpointing (2 балла)

**Activation checkpointing** (gradient checkpointing) во время forward pass **не сохраняет активации промежуточных слоёв** в памяти GPU. Вместо этого на backward pass они пересчитываются заново.

- **Экономия памяти**: из экспериментов — **1.19 GB** (5.25 GB → 4.06 GB). Это соответствует освобождению большей части буферов активаций.
- **Цена**: дополнительный partial forward pass на backward → throughput упал на **−19.0%** (87 197 → 70 612 tok/s). Это укладывается в ожидаемый диапазон ~20–33%.

**Теоретическое обоснование:** для transformer с $L$ слоями без AC хранится $O(L \times B \times S \times H)$ активаций. С AC — $O(\sqrt{L} \times B \times S \times H)$ при оптимальной стратегии сохранения чекпоинтов. Платой является один дополнительный forward pass для каждого чекпоинтированного блока.


In [ ]:
# 1.B — Throughput comparison
tps_bf16 = float(r_bf16.get('throughput_tps', 0))
tps_bf16ac = float(r_bf16ac.get('throughput_tps', 0))
mem_bf16_v = float(r_bf16.get('peak_mem_gb', 0))
mem_bf16ac_v = float(r_bf16ac.get('peak_mem_gb', 0))

if tps_bf16 > 0 and tps_bf16ac > 0:
    delta_tps = (tps_bf16ac - tps_bf16) / tps_bf16 * 100
    delta_mem = mem_bf16_v - mem_bf16ac_v
    print(f'bf16 throughput:    {tps_bf16:.0f} tok/s')
    print(f'bf16+AC throughput: {tps_bf16ac:.0f} tok/s')
    print(f'Δ throughput:       {delta_tps:+.1f}%  (expected ~-20 to -30%)')
    print(f'\nbf16 peak mem:    {mem_bf16_v:.3f} GB')
    print(f'bf16+AC peak mem: {mem_bf16ac_v:.3f} GB')
    print(f'Memory saved by AC: {delta_mem:.3f} GB')

### 1.C — Val PPL: совпадают ли? (1 балл)

Из экспериментов:
- **fp32**: val PPL = 66.96
- **bf16**: val PPL = 108.08
- **bf16+AC**: val PPL = 109.19

**fp32 vs bf16**: расхождение составило **41.12 единицы** — это очень большое и неожиданное расхождение. Такая разница объясняется **разными гиперпараметрами сходимости** при разных dtype: fp32 обучается медленнее (в 2× меньше throughput), но модель видит те же данные за больше реального времени. Возможно, learning rate или warmup были оптимизированы под bf16. В теории при одинаковых гиперпараметрах fp32 и bf16 должны давать схожий PPL — расхождение > 1 единицы нежелательно и указывает на численную несовместимость конфигурации.

**bf16 vs bf16+AC**: расхождение **1.11 единицы** — на границе приемлемого. Activation checkpointing является математически эквивалентной операцией и **не должен влиять на val PPL**. Небольшое отклонение может быть вызвано пересчётом активаций в bf16 с другим порядком операций (разная численная точность при recompute).


In [ ]:
# 1.C — PPL comparison
for label, r in [('fp32', r_fp32), ('bf16', r_bf16), ('bf16+AC', r_bf16ac)]:
    ppl = r.get('val_ppl', '—')
    print(f'{label:<25}: val_ppl = {ppl}')

ppl_fp32 = float(r_fp32.get('val_ppl', 0))
ppl_bf16 = float(r_bf16.get('val_ppl', 0))
ppl_bf16ac = float(r_bf16ac.get('val_ppl', 0))
if ppl_bf16 > 0 and ppl_bf16ac > 0:
    diff_fp32_bf16 = abs(ppl_fp32 - ppl_bf16)
    diff_bf16_ac = abs(ppl_bf16 - ppl_bf16ac)
    print(f'\n|fp32 PPL - bf16 PPL| = {diff_fp32_bf16:.4f}  (large — dtype affects convergence)')
    print(f'|bf16 PPL - bf16+AC PPL| = {diff_bf16_ac:.4f}  (AC is mathematically equivalent, small diff expected)')

---
## 2. FSDP: стратегии шардирования


In [ ]:
r2_no  = load_results('p2-no-shard')
r2_sgo = load_results('p2-shard-grad-op')
r2_fs  = load_results('p2-full-shard')

print('Task 2 — FSDP Sharding Strategies (4 GPU, bf16)')
print(f"{'Strategy':<18} {'Val PPL':>10} {'Peak mem/GPU (GB)':>18} {'Throughput (tok/s)':>20}")
print('-' * 70)
for label, r in [('NO_SHARD (DDP)', r2_no), ('SHARD_GRAD_OP', r2_sgo), ('FULL_SHARD', r2_fs)]:
    print(f"{label:<18} {fmt(r,'val_ppl'):>10} {fmt(r,'peak_mem_gpu_gb'):>18} {fmt(r,'throughput_tps',0):>20}")

### 2.A — Теоретический расход памяти (2 балла)

Параметры модели: $P = 160M$. Обучение с bf16, optimizer states в fp32.

**Состояния на шаге обучения:**

| Что | dtype | Размер |
|---|---|---|
| Параметры | bf16 | $2P$ байт |
| Градиенты | bf16/fp32 | $2P$ байт |
| Adam momentum | fp32 | $4P$ байт |
| Adam variance | fp32 | $4P$ байт |
| **Итого model states** | | **$12P$ байт** |

Для $P = 160\text{M}$: $12 \times 160\text{M} = 1.92$ GB (без активаций).

**По стратегиям на 4 GPU:**

- **NO_SHARD (DDP)**: полная копия на каждой GPU → **≈ 1.92 GB** model states/GPU
- **SHARD_GRAD_OP**: шардируются только gradients + optimizer states, параметры реплицированы → **≈ 2P + (2+4+4)P/4 = 0.32 + 0.40 = 0.72 GB**
- **FULL_SHARD**: всё шардируется → **≈ 12P/4 = 0.48 GB** model states/GPU

**Реальные vs теоретические:**
- NO_SHARD: теория 1.92 GB → измерено **3.59 GB** (overhead +1.67 GB — активации + CUDA context + NCCL буферы)
- SHARD_GRAD_OP: теория 0.72 GB → измерено **2.67 GB** (overhead +1.95 GB)
- FULL_SHARD: теория 0.48 GB → измерено **2.52 GB** (overhead +2.04 GB)

Overhead выше для FULL_SHARD из-за дополнительных AllGather-буферов. Активации не шардируются → примерно константный overhead ~1.7–2.0 GB.


In [ ]:
# 2.A — Theoretical vs measured memory
n_params = 160e6
n_gpus = 4

# bytes per param: 2(param bf16) + 2(grad bf16) + 4(mom fp32) + 4(var fp32) = 12
bytes_per_param_total = 12
total_model_states_gb = bytes_per_param_total * n_params / 1e9

# Theoretical per-GPU
theo = {
    'NO_SHARD':       total_model_states_gb,          # fully replicated
    'SHARD_GRAD_OP':  (2 * n_params / 1e9) + ((2+4+4) * n_params / 1e9) / n_gpus,  # params replicated, rest sharded
    'FULL_SHARD':     total_model_states_gb / n_gpus,  # everything sharded
}

print('Memory analysis (model states only, no activations):')
print(f"{'Strategy':<18} {'Theoretical (GB)':>18} {'Measured (GB)':>15} {'Overhead (GB)':>15}")
print('-' * 70)
for label, (key, r) in [('NO_SHARD', ('NO_SHARD', r2_no)),
                         ('SHARD_GRAD_OP', ('SHARD_GRAD_OP', r2_sgo)),
                         ('FULL_SHARD', ('FULL_SHARD', r2_fs))]:
    t = theo[key]
    m = float(r.get('peak_mem_gpu_gb', 0))
    overhead = m - t if m > 0 else float('nan')
    print(f"{label:<18} {t:>18.3f} {m:>15.3f} {overhead:>15.3f}")

### 2.B — Throughput FULL_SHARD vs NO_SHARD (2 балла)

**Коммуникации в каждой стратегии:**

| Стратегия | Forward | Backward | Optimizer step |
|---|---|---|---|
| NO_SHARD (DDP) | нет | AllReduce(grads) 1× | локально |
| SHARD_GRAD_OP | AllGather(params) 1× | ReduceScatter(grads) | локально |
| FULL_SHARD | AllGather(params) на каждом слое | ReduceScatter(grads) на каждом слое | локально |

**Измеренный результат:** FULL_SHARD throughput = **86 035 tok/s** vs NO_SHARD = **153 631 tok/s** → **−44%** (FULL_SHARD/NO_SHARD = 0.56×).

FULL_SHARD выполняет **AllGather + ReduceScatter на каждый слой** (12 слоёв у pythia-160m), тогда как DDP — только 1 AllReduce по всем параметрам. Для малой модели (160M) время вычислений невелико и его недостаточно для перекрытия коммуникаций → overhead значителен.

**Для модели в 10× больше (1.6B)**: каждый слой шире (больше FLOPs), время вычислений растёт квадратично с hidden_dim, тогда как коммуникации — линейно. Поэтому **relative overhead FULL_SHARD уменьшится** и throughput ratio приблизится к 1.0.


In [ ]:
# 2.B — Throughput ratio
tps_no   = float(r2_no.get('throughput_tps', 0))
tps_sgo  = float(r2_sgo.get('throughput_tps', 0))
tps_full = float(r2_fs.get('throughput_tps', 0))

if tps_no > 0 and tps_full > 0:
    ratio = tps_full / tps_no
    print(f'NO_SHARD throughput:   {tps_no:.0f} tok/s')
    print(f'SHARD_GRAD_OP:         {tps_sgo:.0f} tok/s')
    print(f'FULL_SHARD throughput: {tps_full:.0f} tok/s')
    print(f'FULL_SHARD / NO_SHARD: {ratio:.3f}x  ({(ratio-1)*100:+.1f}%)')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

strategies = ['NO_SHARD\n(DDP)', 'SHARD_GRAD_OP', 'FULL_SHARD']
mems   = [float(r.get('peak_mem_gpu_gb', 0)) for r in [r2_no, r2_sgo, r2_fs]]
tps_all = [float(r.get('throughput_tps', 0)) for r in [r2_no, r2_sgo, r2_fs]]
colors = ['#4C72B0', '#DD8452', '#55A868']

axes[0].bar(strategies, mems, color=colors, alpha=0.8, edgecolor='k')
axes[0].set_title('Peak mem/GPU — Task 2 (4 GPU)')
axes[0].set_ylabel('GB')

axes[1].bar(strategies, tps_all, color=colors, alpha=0.8, edgecolor='k')
axes[1].set_title('Throughput — Task 2 (4 GPU)')
axes[1].set_ylabel('tok/s')

plt.tight_layout()
plt.savefig('plots/task2_bars.png', bbox_inches='tight')
plt.show()

### 2.C — Val PPL совпадают? (1 балл)

Из экспериментов:
- NO_SHARD: 110.98
- SHARD_GRAD_OP: 110.91  
- FULL_SHARD: 110.57

Расхождение между стратегиями **< 0.5 единиц** — теоретически все три стратегии математически эквивалентны при одинаковом `global_batch_size` и `seed`. Наблюдаемые минимальные отличия укладываются в норму численной погрешности bf16 при разном порядке коммуникаций.


In [ ]:
for label, r in [('NO_SHARD', r2_no), ('SHARD_GRAD_OP', r2_sgo), ('FULL_SHARD', r2_fs)]:
    print(f'{label:<20}: val_ppl = {r.get("val_ppl", "—")}')

---
## 3. CPU Offload


In [ ]:
r3_fs    = load_results('p3-full-shard')
r3_co    = load_results('p3-cpu-offload')
r3_coac  = load_results('p3-cpu-offload-ac')

print('Task 3 — CPU Offload (2 GPU, FULL_SHARD, bf16)')
print(f"{'Config':<40} {'Peak GPU mem (GB)':>18} {'Throughput (tok/s)':>20}")
print('-' * 80)
configs = [
    ('FULL_SHARD', r3_fs),
    ('FULL_SHARD + CPUOffload', r3_co),
    ('FULL_SHARD + CPUOffload + AC', r3_coac),
]
for label, r in configs:
    print(f"{label:<40} {fmt(r,'peak_mem_gpu_gb'):>18} {fmt(r,'throughput_tps',0):>20}")

### 3.A — Что делает CPUOffload? (2 балла)

**CPUOffload** перемещает **optimizer states (momentum + variance) и параметры** из GPU RAM в CPU RAM:

- **На CPU постоянно:** optimizer states (fp32 momentum + variance = 8P байт)
- **На GPU только во время forward/backward:** параметры (переносятся на GPU, используются, переносятся обратно)
- **Всегда на GPU:** активации текущего слоя, градиенты

**Из экспериментов (2 GPU):** память снизилась с **2.82 GB → 2.21 GB**, экономия = **0.61 GB**.  
Теоретическая ожидаемая экономия от optimizer states: 8P / 2 GPU = 8 × 160M / 2 / 1e9 = 0.64 GB — хорошее соответствие.

### 3.B — Совместный эффект CPUOffload + AC (2 балла)

Добавление AC дополнительно снизило память: **2.21 GB → 1.61 GB**, экономия = **0.60 GB**.  

Эффекты **приблизительно аддитивны**: CPUOffload экономит **optimizer states** (~0.61 GB), AC экономит **активации** (~0.60 GB). Эти два пула памяти независимы, поэтому суммарный эффект ≈ аддитивен. Небольшое отклонение от идеальной аддитивности обусловлено общим бюджетом аллокатора PyTorch.

### 3.C — Throughput с CPUOffload (1 балл)

Throughput **упал на −51.7%** при включении CPUOffload (44 763 → 21 599 tok/s).  

Причины:
- CPU↔GPU трафик для параметров на каждом forward шаге
- CPU↔GPU трафик для optimizer states на каждом optimizer step
- Пропускная способность PCIe (≈ 16–32 GB/s) значительно ниже HBM (> 1 TB/s)

Добавление +AC изменило throughput незначительно (−2.1%), поскольку bottleneck уже на CPU-GPU transfer, а не на памяти активаций.


In [ ]:
# 3.A/B/C — Memory and throughput analysis
mem_fs   = float(r3_fs.get('peak_mem_gpu_gb', 0))
mem_co   = float(r3_co.get('peak_mem_gpu_gb', 0))
mem_coac = float(r3_coac.get('peak_mem_gpu_gb', 0))

tps_fs   = float(r3_fs.get('throughput_tps', 0))
tps_co   = float(r3_co.get('throughput_tps', 0))
tps_coac = float(r3_coac.get('throughput_tps', 0))

if mem_fs and mem_co:
    print(f'Memory saved by CPUOffload: {mem_fs - mem_co:.3f} GB')
    print(f'  Expected (optimizer states only): {8 * 160e6 / 1e9 / 2:.3f} GB (on 2 GPU)')
if mem_co and mem_coac:
    print(f'Additional memory saved by +AC:    {mem_co - mem_coac:.3f} GB')
if tps_fs and tps_co:
    print(f'\nThroughput change with CPUOffload: {(tps_co/tps_fs - 1)*100:+.1f}%')
if tps_co and tps_coac:
    print(f'Throughput change with +AC:        {(tps_coac/tps_co - 1)*100:+.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ['FULL_SHARD', 'FULL_SHARD\n+CPUOffload', 'FULL_SHARD\n+CPUOffload+AC']
axes[0].bar(labels, [mem_fs, mem_co, mem_coac], color=['#4C72B0','#DD8452','#55A868'],
            alpha=0.8, edgecolor='k')
axes[0].set_title('Peak GPU Memory — Task 3 (2 GPU)')
axes[0].set_ylabel('GB')

axes[1].bar(labels, [tps_fs, tps_co, tps_coac], color=['#4C72B0','#DD8452','#55A868'],
            alpha=0.8, edgecolor='k')
axes[1].set_title('Throughput — Task 3 (2 GPU)')
axes[1].set_ylabel('tok/s')

plt.tight_layout()
plt.savefig('plots/task3_bars.png', bbox_inches='tight')
plt.show()

---
## 4. Масштабирование по числу GPU


In [ ]:
r4_1 = load_results('p4-1gpu')
r4_2 = load_results('p4-2gpu')
r4_4 = load_results('p4-4gpu')

print('Task 4 — GPU Scaling (FULL_SHARD, bf16)')
print(f"{'N GPU':>6} {'Throughput total (tok/s)':>26} {'Throughput per GPU (tok/s)':>28} {'Peak mem/GPU (GB)':>18}")
print('-' * 82)
for n, r in [(1, r4_1), (2, r4_2), (4, r4_4)]:
    tps_total = float(r.get('throughput_tps', 0))
    tps_per = float(r.get('throughput_per_gpu_tps', tps_total / n if tps_total else 0))
    mem = fmt(r, 'peak_mem_gpu_gb')
    print(f"{n:>6} {tps_total:>26.0f} {tps_per:>28.0f} {mem:>18}")

### 4.A — Scaling efficiency (3 балла)

**Scaling efficiency:**
$$\eta(N) = \frac{\text{throughput\_per\_gpu}(N)}{\text{throughput\_per\_gpu}(1)}$$

Из экспериментов:
- 1 GPU: throughput/GPU = 87 248 tok/s (baseline, FULL_SHARD без коммуникаций)
- 2 GPU: throughput/GPU = 20 098 tok/s → **η(2) = 0.23**
- 4 GPU: throughput/GPU = 9 481 tok/s → **η(4) = 0.11**

Эффективность масштабирования крайне низкая. Причины:
1. **Коммуникационный overhead FULL_SHARD**: на каждом слое 2 AllGather + ReduceScatter. Для pythia-160m вычисления слоёв быстрые, коллективы не перекрываются.
2. **NCCL sync overhead** растёт с числом GPU.
3. **1 GPU с FULL_SHARD**: фактически нет коммуникаций → искусственно высокий baseline.

Для больших моделей (7B+) η значительно выше — вычисления доминируют над коммуникациями.


In [ ]:
# 4.A — Scaling efficiency
tps_1gpu = float(r4_1.get('throughput_per_gpu_tps',
                           r4_1.get('throughput_tps', 0)))
tps_2gpu = float(r4_2.get('throughput_per_gpu_tps',
                           float(r4_2.get('throughput_tps', 0)) / 2))
tps_4gpu = float(r4_4.get('throughput_per_gpu_tps',
                           float(r4_4.get('throughput_tps', 0)) / 4))

n_list = [1, 2, 4]
tps_list = [tps_1gpu, tps_2gpu, tps_4gpu]
eta_list = [t / tps_1gpu if tps_1gpu else 0 for t in tps_list]

for n, tps, eta in zip(n_list, tps_list, eta_list):
    print(f'N={n}: throughput_per_gpu={tps:.0f} tok/s  η={eta:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(n_list, tps_list, 'o-', color='#4C72B0', lw=2, ms=8, label='Measured')
if tps_1gpu:
    axes[0].axhline(tps_1gpu, color='gray', ls='--', label='Ideal (constant)')
axes[0].set_title('Throughput per GPU vs N (Task 4)')
axes[0].set_xlabel('Number of GPUs')
axes[0].set_ylabel('tok/s per GPU')
axes[0].set_xticks(n_list)
axes[0].legend()

axes[1].plot(n_list, eta_list, 's-', color='#DD8452', lw=2, ms=8)
axes[1].axhline(1.0, color='gray', ls='--', label='Perfect scaling η=1')
axes[1].set_title('Scaling Efficiency η(N) (Task 4)')
axes[1].set_xlabel('Number of GPUs')
axes[1].set_ylabel('η = tps_per_gpu(N) / tps_per_gpu(1)')
axes[1].set_xticks(n_list)
axes[1].set_ylim(0, 1.1)
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/task4_scaling.png', bbox_inches='tight')
plt.show()

### 4.B — Memory scaling (2 балла)

Из экспериментов:
- N=1: 5.25 GB
- N=2: 2.82 GB
- N=4: 1.52 GB

Отношение **mem(N=1) / mem(N=4) = 3.45×** — близко к 4, но не равно.

Почему не ровно 4×?

1. **Активации** — не шардируются в FSDP (каждая GPU держит свои активации для своего батча).
2. **CUDA context / буферы NCCL** — фиксированный overhead ~300–500 MB на GPU.
3. **Буферы AllGather** — временные буферы для сборки параметров из шардов.

Model states масштабируются как 1/N, активации — не масштабируются, overhead — константный. Поэтому реальное отношение mem(1)/mem(4) < 4.


In [ ]:
# 4.B — Memory scaling analysis
mem_1 = float(r4_1.get('peak_mem_gpu_gb', 0))
mem_2 = float(r4_2.get('peak_mem_gpu_gb', 0))
mem_4 = float(r4_4.get('peak_mem_gpu_gb', 0))

if mem_1 and mem_4:
    ratio_14 = mem_1 / mem_4
    print(f'mem(N=1) = {mem_1:.3f} GB')
    print(f'mem(N=2) = {mem_2:.3f} GB')
    print(f'mem(N=4) = {mem_4:.3f} GB')
    print(f'\nmem(1)/mem(4) = {ratio_14:.2f}x  (ideal: 4x)')
    non_sharded_estimate = mem_1 - (mem_1 - mem_4) * 4 / 3
    print(f'\nEstimated non-sharded memory (activations + CUDA overhead): {non_sharded_estimate:.3f} GB')

fig, ax = plt.subplots(figsize=(7, 4))
ns = [1, 2, 4]
mems = [mem_1, mem_2, mem_4]
ideal = [mem_1 / n for n in ns]

ax.plot(ns, mems, 'o-', color='#4C72B0', lw=2, ms=8, label='Measured')
ax.plot(ns, ideal, 's--', color='gray', lw=1.5, ms=6, label='Ideal (1/N scaling)')
ax.set_title('Peak Memory/GPU vs N GPUs (Task 4)')
ax.set_xlabel('Number of GPUs')
ax.set_ylabel('GB')
ax.set_xticks(ns)
ax.legend()
plt.tight_layout()
plt.savefig('plots/task4_memory.png', bbox_inches='tight')
plt.show()

---
## Bonus A — Влияние размера модели на overhead шардирования


In [ ]:
# Load bonus results: 160m (from task 2) and 410m
b410_no  = load_results('bonus-410m-no-shard')
b410_sgo = load_results('bonus-410m-shard-grad-op')
b410_fs  = load_results('bonus-410m-full-shard')

# Reuse task 2 results for 160m
b160_no  = r2_no
b160_sgo = r2_sgo
b160_fs  = r2_fs

print('Bonus A — 4 GPU comparison: pythia-160m vs pythia-410m')
strategies = ['NO_SHARD', 'SHARD_GRAD_OP', 'FULL_SHARD']
rows_160 = [b160_no, b160_sgo, b160_fs]
rows_410 = [b410_no, b410_sgo, b410_fs]

print(f"\n{'Strategy':<18} {'160m mem':>10} {'410m mem':>10} {'160m tps':>10} {'410m tps':>10}")
print('-' * 62)
for s, r160, r410 in zip(strategies, rows_160, rows_410):
    print(f"{s:<18} {fmt(r160,'peak_mem_gpu_gb'):>10} {fmt(r410,'peak_mem_gpu_gb'):>10} "
          f"{fmt(r160,'throughput_tps',0):>10} {fmt(r410,'throughput_tps',0):>10}")

### A.A — Для какой модели overhead FULL_SHARD больше? (3 балла)

Из экспериментов:
- **160m**: NO_SHARD = 153 631 tok/s, FULL_SHARD = 86 035 tok/s → overhead = **44.0%**
- **410m**: NO_SHARD = 45 328 tok/s, FULL_SHARD = 29 468 tok/s → overhead = **35.0%**

**Относительный overhead FULL_SHARD больше для 160m.**

**Почему**: При FULL_SHARD каждый AllGather пересылает весь шард параметров слоя. Объём коммуникаций растёт **линейно** с числом параметров, а вычисления (FLOPs) — пропорционально **hidden_dim²** (для матричных умножений). У pythia-410m шире слои → больше FLOPs на шаг → лучше arithmetic intensity → коммуникации лучше перекрываются вычислениями → relative overhead меньше.

### A.B — Экономия памяти NO_SHARD → FULL_SHARD (3 балла)

Из экспериментов:
- **160m**: NO_SHARD = 3.59 GB, FULL_SHARD = 2.52 GB → saved = **1.07 GB (29.8%)**
- **410m**: NO_SHARD = 5.36 GB, FULL_SHARD = 2.56 GB → saved = **2.80 GB (52.2%)**

Отношение экономии: 2.80 / 1.07 ≈ **2.62×**, отношение числа параметров: 410/160 ≈ **2.56×** — хорошее соответствие!

Экономия соответствует соотношению размеров моделей, что подтверждает: при шардировании экономится именно memory, пропорциональная числу параметров (model states = params + grads + optimizer states). Небольшое отклонение от точного соотношения 2.56× объясняется константным overhead (CUDA context, активации, NCCL буферы), который одинаков для обеих моделей.


In [ ]:
# Bonus A — Comparative plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(strategies))
w = 0.35

# Peak mem comparison
mems_160 = [float(r.get('peak_mem_gpu_gb', 0)) for r in rows_160]
mems_410 = [float(r.get('peak_mem_gpu_gb', 0)) for r in rows_410]

axes[0].bar(x - w/2, mems_160, w, label='pythia-160m', color='#4C72B0', alpha=0.8, edgecolor='k')
axes[0].bar(x + w/2, mems_410, w, label='pythia-410m', color='#DD8452', alpha=0.8, edgecolor='k')
axes[0].set_title('Peak mem/GPU — 160m vs 410m (4 GPU)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(strategies)
axes[0].set_ylabel('GB')
axes[0].legend()

# Throughput comparison
tps_160 = [float(r.get('throughput_tps', 0)) for r in rows_160]
tps_410 = [float(r.get('throughput_tps', 0)) for r in rows_410]

axes[1].bar(x - w/2, tps_160, w, label='pythia-160m', color='#4C72B0', alpha=0.8, edgecolor='k')
axes[1].bar(x + w/2, tps_410, w, label='pythia-410m', color='#DD8452', alpha=0.8, edgecolor='k')
axes[1].set_title('Throughput — 160m vs 410m (4 GPU)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(strategies)
axes[1].set_ylabel('tok/s')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/bonus_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# A.A — Relative overhead of FULL_SHARD vs NO_SHARD for each model
for model_label, r_no, r_fs in [
    ('160m', b160_no, b160_fs),
    ('410m', b410_no, b410_fs)
]:
    tps_no = float(r_no.get('throughput_tps', 0))
    tps_fs = float(r_fs.get('throughput_tps', 0))
    if tps_no > 0 and tps_fs > 0:
        overhead = (1 - tps_fs / tps_no) * 100
        print(f'{model_label}: FULL_SHARD overhead = {overhead:.1f}% vs NO_SHARD')
        print(f'         (NO_SHARD: {tps_no:.0f} tok/s  FULL_SHARD: {tps_fs:.0f} tok/s)')

# A.B — Memory savings NO_SHARD -> FULL_SHARD
print()
for model_label, r_no, r_fs in [
    ('160m', b160_no, b160_fs),
    ('410m', b410_no, b410_fs)
]:
    mem_no = float(r_no.get('peak_mem_gpu_gb', 0))
    mem_fs = float(r_fs.get('peak_mem_gpu_gb', 0))
    if mem_no and mem_fs:
        saved = mem_no - mem_fs
        pct = saved / mem_no * 100
        print(f'{model_label}: memory saved NO_SHARD→FULL_SHARD = {saved:.3f} GB ({pct:.1f}%)')

# Ratio comparison
saved_160 = float(b160_no.get('peak_mem_gpu_gb', 0)) - float(b160_fs.get('peak_mem_gpu_gb', 0))
saved_410 = float(b410_no.get('peak_mem_gpu_gb', 0)) - float(b410_fs.get('peak_mem_gpu_gb', 0))
if saved_160 > 0:
    print(f'\nRatio of memory savings 410m/160m: {saved_410/saved_160:.2f}x')
    print(f'Ratio of model parameters 410/160: {410/160:.2f}x')
    print('→ Savings scale approximately proportionally to model size')

---
## Итоговые таблицы


In [ ]:
print('=' * 80)
print('TASK 1 — Single-GPU Baseline')
print('=' * 80)
print(f"{'Config':<35} {'Val PPL':>10} {'Peak mem (GB)':>14} {'Throughput (tok/s)':>20}")
print('-' * 81)
for label, r in [('fp32', r_fp32), ('bf16', r_bf16), ('bf16 + activation checkpointing', r_bf16ac)]:
    print(f"{label:<35} {fmt(r,'val_ppl'):>10} {fmt(r,'peak_mem_gb'):>14} {fmt(r,'throughput_tps',0):>20}")

print()
print('=' * 80)
print('TASK 2 — FSDP Strategies (4 GPU, bf16)')
print('=' * 80)
print(f"{'Strategy':<18} {'Val PPL':>10} {'Peak mem/GPU':>14} {'Throughput (tok/s)':>20}")
print('-' * 64)
for label, r in [('NO_SHARD', r2_no), ('SHARD_GRAD_OP', r2_sgo), ('FULL_SHARD', r2_fs)]:
    print(f"{label:<18} {fmt(r,'val_ppl'):>10} {fmt(r,'peak_mem_gpu_gb'):>14} {fmt(r,'throughput_tps',0):>20}")

print()
print('=' * 80)
print('TASK 3 — CPU Offload (2 GPU, FULL_SHARD, bf16)')
print('=' * 80)
print(f"{'Config':<38} {'Peak GPU mem (GB)':>18} {'Throughput (tok/s)':>20}")
print('-' * 78)
for label, r in [('FULL_SHARD', r3_fs), ('FULL_SHARD+CPUOffload', r3_co), ('FULL_SHARD+CPUOffload+AC', r3_coac)]:
    print(f"{label:<38} {fmt(r,'peak_mem_gpu_gb'):>18} {fmt(r,'throughput_tps',0):>20}")

print()
print('=' * 80)
print('TASK 4 — Scaling (FULL_SHARD, bf16)')
print('=' * 80)
print(f"{'N GPU':>6} {'Throughput total':>18} {'Throughput/GPU':>16} {'Peak mem/GPU':>14}")
print('-' * 58)
for n, r in [(1, r4_1), (2, r4_2), (4, r4_4)]:
    tps_total = float(r.get('throughput_tps', 0))
    tps_per = float(r.get('throughput_per_gpu_tps', tps_total / n if tps_total else 0))
    print(f"{n:>6} {tps_total:>18.0f} {tps_per:>16.0f} {fmt(r,'peak_mem_gpu_gb'):>14}")